# RANCANGAN ACAK KELOMPOK LENGKAP PADA PENGARUH HARGA BARANG DAN JASA TERHADAP INFLASI

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import zscore
from sklearn.model_selection import train_test_split
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.stats.stattools import durbin_watson
from scipy.stats import kstest, norm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import shapiro, levene

In [2]:
data = pd.read_excel('/content/inflasi_sim_500.xlsx')
print(data.head())

   ID       Kelompok  Inflasi
0   1  Bahan Makanan  -0.3141
1   2  Bahan Makanan   0.2672
2   3  Bahan Makanan   0.0419
3   4  Bahan Makanan   0.7055
4   5  Bahan Makanan   0.0943


In [3]:
data.columns

Index(['ID', 'Kelompok', 'Inflasi'], dtype='object')

In [4]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ID        500 non-null    int64  
 1   Kelompok  500 non-null    object 
 2   Inflasi   500 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 11.8+ KB
None


# PEMBERSIHAN DATA

In [5]:
print(data.isnull().sum())

ID          0
Kelompok    0
Inflasi     0
dtype: int64


In [6]:
print(data.duplicated())

0      False
1      False
2      False
3      False
4      False
       ...  
495    False
496    False
497    False
498    False
499    False
Length: 500, dtype: bool


# UJI ASUMSI

In [12]:
# --- Uji Kolmogorov-Smirnov pada seluruh data ---
data_inflasi = data["Inflasi"]

# Hitung rata-rata dan standar deviasi untuk distribusi normal pembanding
mu, sigma = data_inflasi.mean(), data_inflasi.std(ddof=0)

# KS test: bandingkan distribusi data dengan distribusi normal(mu, sigma)
ks_stat, p_value = kstest(data_inflasi, 'norm', args=(mu, sigma))

print("=== Uji Kolmogorov-Smirnov (Seluruh Data) ===")
print(f"KS Statistic = {ks_stat:.4f}, p-value = {p_value:.4f}")
if p_value > 0.05:
    print("keseluruhan berdistribusi normal ")
else:
    print("keseluruhan tidak berdistribusi normal")

# --- Uji Kolmogorov-Smirnov per perlakuan ---
print("\n=== Uji KS per Perlakuan ===")
for kelompok, subset in data.groupby("Kelompok")["Inflasi"]:
    mu, sigma = subset.mean(), subset.std(ddof=0)
    ks_stat, p_value = kstest(subset, 'norm', args=(mu, sigma))
    print(f"{kelompok}: KS Stat = {ks_stat:.4f}, p-value = {p_value:.4f}")
    if p_value > 0.05:
        print(f"  Data {kelompok} berdistribusi normal")
    else:
        print(f"  Data {kelompok} tidak berdistribusi normal")

=== Uji Kolmogorov-Smirnov (Seluruh Data) ===
KS Statistic = 0.0203, p-value = 0.9832
keseluruhan berdistribusi normal 

=== Uji KS per Perlakuan ===
Bahan Makanan: KS Stat = 0.0563, p-value = 0.9858
  Data Bahan Makanan berdistribusi normal
Kesehatan: KS Stat = 0.0467, p-value = 0.9933
  Data Kesehatan berdistribusi normal
Makanan Jadi, Minuman, Rokok, Tembakau: KS Stat = 0.0679, p-value = 0.9054
  Data Makanan Jadi, Minuman, Rokok, Tembakau berdistribusi normal
Pendidikan, Rekreasi, Olahraga: KS Stat = 0.0610, p-value = 0.9544
  Data Pendidikan, Rekreasi, Olahraga berdistribusi normal
Perumahan, Air, Listrik, Gas, Bahan Bakar: KS Stat = 0.1076, p-value = 0.5355
  Data Perumahan, Air, Listrik, Gas, Bahan Bakar berdistribusi normal
Sandang: KS Stat = 0.1068, p-value = 0.5569
  Data Sandang berdistribusi normal
Transport, Komunikasi, Jasa Keuangan: KS Stat = 0.0677, p-value = 0.9076
  Data Transport, Komunikasi, Jasa Keuangan berdistribusi normal
Umum: KS Stat = 0.0983, p-value = 0.5531

In [14]:
# ----------------------------
# UJI HOMOGENITAS VARIANS (LEVENe)
# ----------------------------
print("\n=== Uji Homogenitas Varians (Levene) ===")
groups = [group["Inflasi"].values for kelompok, group in data.groupby("Kelompok")]
stat, p_value = levene(*groups)
print(f"Statistic = {stat:.4f}, p-value = {p_value:.4f}")
if p_value > 0.05:
    print("→ homogen")
else:
    print("→ hetero")


=== Uji Homogenitas Varians (Levene) ===
Statistic = 0.6605, p-value = 0.7056
→ homogen


# HIPOTESIS
H0 : tidak ada perbedaan signifikan

H1 : paling sedikit ada satu perbedaan signifikan

In [20]:
df = pd.DataFrame(data)
# --- Membuat model ANOVA ---
model = ols(' inflasi ~  C(kelompok) + C(inflasi)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

# --- Tampilkan hasil ---
print("Hasil Uji ANOVA:")
print(anova_table)

PatsyError: Error evaluating factor: NameError: name 'inflasi' is not defined
    inflasi ~  C(kelompok) + C(inflasi)
                             ^^^^^^^^^^